In [27]:
import math
from typing import Optional

SPARSE_M = 4

class OpConfig:
    # Matrix dimensions
    M: int
    N: int
    K: int
    H: Optional[int]          # optional head / batch dimension (e.g., attention)
    cfg_sparse_n: int         # number of non‑zeros per group (1~4, where 4 means dense)
    
    # Systolic array geometry (to be set by the caller)
    bank: int = 1             # number of parallel banks (3‑D array depth)
    rows: int = 16            # array rows (tile height, M dimension)
    cols: int = 16            # array columns (tile width, N dimension)
    

def get_exact_param_flops_cycle(op_cfg: OpConfig):
    """
    Compute compressed parameter count, FLOPs, memory footprint and the exact
    number of cycles required on a banked 3‑D systolic array.
    
    Returns a dict with keys: params, flops, mem_bytes, cycles.
    """
    # ---- Sparse encoding overhead ----
    # byte_per_mask: additional metadata for each sparse group
    if op_cfg.cfg_sparse_n == 4:
        byte_per_mask = 0.0
    elif op_cfg.cfg_sparse_n == 2:
        byte_per_mask = 0.5
    else:
        byte_per_mask = 0.25
    
    weight_per_group = op_cfg.cfg_sparse_n
    is_dense = (op_cfg.cfg_sparse_n == 4)          # 4:4 pattern means fully dense
    
    # ---- Original (uncompressed) parameters and FLOPs ----
    # Attention‑like operations include an extra H dimension
    is_act_act = hasattr(op_cfg, "H") and op_cfg.H is not None
    
    # Original parameter count (weights only, assumes N×K matrix)
    ori_params = (op_cfg.N * op_cfg.K) # * weight_per_group / SPARSE_M
    
    # Original FLOPs (M×N×K × 2 for MAC, but convention often uses M*N*K)
    if not is_act_act:
        ori_flops = op_cfg.M * op_cfg.N * op_cfg.K
    else:
        ori_flops = op_cfg.M * op_cfg.N * op_cfg.K * op_cfg.H
    
    # ---- Compressed quantities (after sparse packing) ----
    compressed_params = (ori_params / SPARSE_M) * weight_per_group
    compressed_flops  = (ori_flops / SPARSE_M) * weight_per_group
    
    # Total memory bytes = packed weights + mask overhead
    total_mem_byte = compressed_params + byte_per_mask * (ori_params / SPARSE_M)
    
    # ---- Systolic cycle modelling ----
    if is_dense:
        reps = op_cfg.K
    else:
        reps = (op_cfg.K+ SPARSE_M - 1) // SPARSE_M   # ceil(tile_k / SPARSE_M)
    
    max_cycles = (1 if op_cfg.cfg_sparse_n == 4 else op_cfg.cfg_sparse_n)
    skew_size = (op_cfg.cols - 1) * max_cycles # (1 if op_cfg.cfg_sparse_n == 4 else max_cycles)
    # skew_size = (op_cfg.rows + op_cfg.cols - 1) * max_cycles # (1 if op_cfg.cfg_sparse_n == 4 else max_cycles)

    print("MAX_CYCLES:", max_cycles, "skew_size:", skew_size)
    skewed_reps = reps * max_cycles + skew_size
    
    tile_m_count = (op_cfg.M + op_cfg.rows - 1) // op_cfg.rows
    tile_n_count = (op_cfg.N + op_cfg.cols - 1) // op_cfg.cols
    
    total_spatial_tiles = tile_m_count * tile_n_count   # number of (M,N) blocks
    
    # 5. Parallel execution on multiple banks
    bank_iters = (total_spatial_tiles + op_cfg.bank - 1) // op_cfg.bank
    total_cycles = bank_iters * skewed_reps # * k_tile_count 
    
    # ---- Return aggregated information ----
    return {
        "params": compressed_params,
        "flops": compressed_flops,
        "mem_bytes": total_mem_byte,
        "cycles": total_cycles
    }

In [28]:
# Define systolic array dimensions (example values)
BANK = 4
ROWS = 28
COLS = 12

FREQ_GHZ = 0.15

# List of test configurations: (M, N, K, H, cfg_sparse_n list)
test_cases = [
    (197, 384, 384, None, [3, 2, 1]),   # M=197 N=384 K=384, sparse 3:4, 2:4, 1:4
    (197, 197, 64, 6, [4]),             # M=197 N=197 K=64 H=6, dense (4:4)
]

print("=" * 80)
print("Systolic Array Configuration: Bank={}, Rows={}, Cols={}".format(
    BANK, ROWS, COLS))
print("=" * 80)

for M, N, K, H, sparse_list in test_cases:
    for sparse_n in sparse_list:
        # Build operation configuration
        cfg = OpConfig()
        cfg.M = M
        cfg.N = N
        cfg.K = K
        cfg.H = H
        cfg.cfg_sparse_n = sparse_n
        
        # Set systolic array geometry
        cfg.bank = BANK
        cfg.rows = ROWS
        cfg.cols = COLS
        
        # Compute parameters, FLOPs, memory and cycles
        result = get_exact_param_flops_cycle(cfg)
        
        gops = result['flops'] * (FREQ_GHZ) / result['cycles']  
        
        # Print results with clear identification
        sparse_type = "dense (4:4)" if sparse_n == 4 else f"{sparse_n}:4 sparse"
        head_info = f" (H={H})" if H is not None else ""
        print(f"\nM={M}, N={N}, K={K}{head_info}, {sparse_type}:")
        print(f"  Compressed params : {result['params']:,.1f}")
        print(f"  Compressed FLOPs  : {result['flops']:,.1f}")
        print(f"  Memory bytes      : {result['mem_bytes']:,.1f}")
        print(f"  Estimated cycles  : {result['cycles']:,}")
        print(f"  Estimated GOPs  : {gops}")

Systolic Array Configuration: Bank=4, Rows=28, Cols=12
MAX_CYCLES: 3 skew_size: 33

M=197, N=384, K=384, 3:4 sparse:
  Compressed params : 110,592.0
  Compressed FLOPs  : 21,786,624.0
  Memory bytes      : 119,808.0
  Estimated cycles  : 20,544
  Estimated GOPs  : 159.0728971962617
MAX_CYCLES: 2 skew_size: 22

M=197, N=384, K=384, 2:4 sparse:
  Compressed params : 73,728.0
  Compressed FLOPs  : 14,524,416.0
  Memory bytes      : 92,160.0
  Estimated cycles  : 13,696
  Estimated GOPs  : 159.07289719626166
MAX_CYCLES: 1 skew_size: 11

M=197, N=384, K=384, 1:4 sparse:
  Compressed params : 36,864.0
  Compressed FLOPs  : 7,262,208.0
  Memory bytes      : 46,080.0
  Estimated cycles  : 6,848
  Estimated GOPs  : 159.07289719626166
MAX_CYCLES: 1 skew_size: 11

M=197, N=197, K=64 (H=6), dense (4:4):
  Compressed params : 12,608.0
  Compressed FLOPs  : 14,902,656.0
  Memory bytes      : 12,608.0
  Estimated cycles  : 2,550
  Estimated GOPs  : 876.6268235294117
